# Spike sorting
&#x26a0;&#xfe0f; **Note** &#x26a0;&#xfe0f;

This notebook demonstrates a spike sorting workflow with kilosort4.

- Spike sorting is computationally intensive and typically benefits from a GPU. Currently, spikeinterface supports NVIDIA GPUs.
- For GPU, please install the cuda environment, `conda env create -f environment_cuda.yml`
- Running this pipeline on CPU is possible but may be slow.
- The dateset used for this notebook is ~3GB.

In [ ]:
from pathlib import Path

from neurokinematics.ephys.spikes.sorting import sort

from huggingface_hub import snapshot_download
import torch

## CUDA availability
If you've created the `neurokinematics_cuda` environment from `environment_cuda.yml`, first check that CUDA is available. You can do this step if using the plain `neurokinematics` environment as well. If unavailable, kilosort will run on the CPU (much slower).

In [ ]:
# check cuda availability
if torch.cuda.is_available():
    ngpus = torch.cuda.device_count()
    print(f"CUDA available: {ngpus} GPU(s)")
    
    for i in range(ngpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

else:
    print("WARNING: CUDA unavailable. Spike sorting will run on CPU.")



## Download open ephys data from huggingface hub
Again, this dataset will be roughly ~3GB, so this will take some time to download.

In [ ]:
# create directory if
data_dir = Path.cwd() / 'sample_oephys_data'

hugging_face_repo = 'cjblack1111/sample_oephys_data'
repo_type = 'dataset'

if not any(data_dir.glob('*')):
    # download entire repo
    print(f"Downloading dataset (~3GB) to {data_dir.resolve()}, this may take a while...")
    oephys_path = snapshot_download(
        repo_id = "cjblack1111/sample_oephys_data",
        repo_type = repo_type,
        local_dir = data_dir,
        local_dir_use_symlinks = False
    )

## Run spike sorting
This recording was performed with an H5 probe from Cambridge Neurotech, so sorting will use the relevant sorting config from the configs folder.

In [ ]:
# use default spike sorting cfg
cfg_file = 'sample_spike_sorting_cfg.yaml'
data_path = data_dir / 'Record Node 109' / 'experiment1' / 'recording1' # set data path toward .oebin - this will depend on which record node your spike filtered data was recorded on

# sort spikes
sorting, recording, probe, analyzer = sort(data_path=data_path, cfg_file=cfg_file)